# Proyecto Práctico: Procesamiento de datos con Spark

Este proyecto aplica operaciones de transformación, integración, filtrado,
agregación y persistencia de datos utilizando PySpark.

El objetivo es trabajar con dos fuentes de información, preparar sus datos,
integrarlas mediante una clave común y generar resultados agregados que
posteriormente serán almacenados en distintos formatos.

## 1. Configuración del entorno

In [ ]:
import os
import urllib.request

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

Entorno Docker correcto


In [4]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PySpark Data Transformation Project")
    .config(
        "spark.hadoop.fs.defaultFS",
        "hdfs://namenode:8020"
    )
    .config(
        "spark.sql.warehouse.dir",
        "hdfs://namenode:8020/user/hive/warehouse"
    )
    .config(
        "spark.driver.extraJavaOptions",
        "-Dderby.system.home=/tmp/derby"
    )
    .enableHiveSupport()
    .getOrCreate()
)

print("Spark:", spark.version)
print(
    "Hadoop:",
    spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 02:50:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0
Hadoop: 3.5.0


## 2. Carga de los datasets

Las rutas de los archivos se obtienen desde variables de entorno para evitar
almacenarlas directamente en el código público debido a la sensbilidad de los datos.

In [8]:
dataset1_url = os.getenv("DATASET_1_URL")
dataset2_url = os.getenv("DATASET_2_URL")

assert dataset1_url, "DATASET_1_URL no está configurada."
assert dataset2_url, "DATASET_2_URL no está configurada."

In [9]:
dataset1_local_path = "/tmp/dataset1.csv"
dataset2_local_path = "/tmp/dataset2.csv"

if not os.path.exists(dataset1_local_path):
    urllib.request.urlretrieve(
        dataset1_url,
        dataset1_local_path
    )

if not os.path.exists(dataset2_local_path):
    urllib.request.urlretrieve(
        dataset2_url,
        dataset2_local_path
    )

In [12]:
dataset1_path = "file:///tmp/dataset1.csv"
dataset2_path = "file:///tmp/dataset2.csv"

df1 = spark.read.csv(
    dataset1_path,
    header=True,
    inferSchema=True
)

df2 = spark.read.csv(
    dataset2_path,
    header=True,
    inferSchema=True
)

In [13]:
df1.show(5)
df2.show(5)

+-----------+-----------+------+-----------+--------+
|customer_id|date_column|amount|description|location|
+-----------+-----------+------+-----------+--------+
|          1|   1/1/2022|  5000| Purchase A| Store A|
|          2|  15/2/2022|  1200| Purchase B| Store B|
|          3|  20/3/2022|   800| Purchase C| Store C|
|          4|  10/4/2022|  3000| Purchase D| Store D|
|          5|   5/5/2022|  6000| Purchase E| Store E|
+-----------+-----------+------+-----------+--------+
only showing top 5 rows
+-----------+----------------+-----+------+
|customer_id|transaction_date|value| notes|
+-----------+----------------+-----+------+
|          1|        1/1/2022| 1500|Note 1|
|          2|       15/2/2022| 2000|Note 2|
|          3|       20/3/2022| 1000|Note 3|
|          4|       10/4/2022| 2500|Note 4|
|          5|        5/5/2022| 1800|Note 5|
+-----------+----------------+-----+------+
only showing top 5 rows


## 3. Exploración de los esquemas

Se revisa la estructura de ambos DataFrames para conocer los nombres de las
columnas y los tipos de datos inferidos por Spark.

In [14]:
df1.printSchema()
df2.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- date_column: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- location: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- value: integer (nullable = true)
 |-- notes: string (nullable = true)



## 4. Creación de variables temporales

Para facilitar el análisis temporal, se convierten las columnas de fecha al tipo `date` de Spark.

A partir de estas fechas se generan dos nuevas variables:

- `year`: año de la transacción en el primer DataFrame.
- `quarter`: trimestre correspondiente al registro en el segundo DataFrame.

Estas variables permitirán realizar posteriormente agregaciones por periodo.

In [15]:
# Convertir la fecha y obtener el año en df1
df1 = (
    df1
    .withColumn(
        "date_column",
        F.to_date("date_column", "d/M/yyyy")
    )
    .withColumn(
        "year",
        F.year("date_column")
    )
)

# Convertir la fecha y obtener el trimestre en df2
df2 = (
    df2
    .withColumn(
        "transaction_date",
        F.to_date("transaction_date", "d/M/yyyy")
    )
    .withColumn(
        "quarter",
        F.quarter("transaction_date")
    )
)

In [16]:
df1.select("date_column", "year").show(5)
df2.select("transaction_date", "quarter").show(5)

+-----------+----+
|date_column|year|
+-----------+----+
| 2022-01-01|2022|
| 2022-02-15|2022|
| 2022-03-20|2022|
| 2022-04-10|2022|
| 2022-05-05|2022|
+-----------+----+
only showing top 5 rows
+----------------+-------+
|transaction_date|quarter|
+----------------+-------+
|      2022-01-01|      1|
|      2022-02-15|      1|
|      2022-03-20|      1|
|      2022-04-10|      2|
|      2022-05-05|      2|
+----------------+-------+
only showing top 5 rows


## 5. Renombrado de columnas

Se renombran las columnas relacionadas con los valores transaccionales para que sus nombres sean más descriptivos y fáciles de interpretar durante las siguientes etapas del análisis.

In [17]:
# Renombrar la columna amount en df1
df1 = df1.withColumnRenamed(
    "amount",
    "transaction_amount"
)

# Renombrar la columna value en df2
df2 = df2.withColumnRenamed(
    "value",
    "transaction_value"
)

In [18]:
print(df1.columns)
print(df2.columns)

['customer_id', 'date_column', 'transaction_amount', 'description', 'location', 'year']
['customer_id', 'transaction_date', 'transaction_value', 'notes', 'quarter']


## 6. Eliminación de columnas innecesarias

Se eliminan las columnas que no serán utilizadas en las siguientes etapas del análisis.

Esta limpieza permite trabajar con DataFrames más simples y mantener únicamente la información relevante para la integración y las agregaciones posteriores.

In [19]:
# Eliminar columnas innecesarias de df1
df1 = df1.drop(
    "description",
    "location"
)

# Eliminar columna innecesaria de df2
df2 = df2.drop(
    "notes"
)

In [20]:
print(df1.columns)
print(df2.columns)

['customer_id', 'date_column', 'transaction_amount', 'year']
['customer_id', 'transaction_date', 'transaction_value', 'quarter']


## 7. Integración de los DataFrames

Se integran ambos DataFrames utilizando `customer_id` como clave común.

La unión permite consolidar en una sola estructura la información proveniente de las dos fuentes para realizar posteriormente filtros y agregaciones sobre los datos combinados.

In [21]:
joined_df = df1.join(
    df2,
    on="customer_id",
    how="inner"
)

In [22]:
joined_df.show(5)

+-----------+-----------+------------------+----+----------------+-----------------+-------+
|customer_id|date_column|transaction_amount|year|transaction_date|transaction_value|quarter|
+-----------+-----------+------------------+----+----------------+-----------------+-------+
|          1| 2022-01-01|              5000|2022|      2022-01-01|             1500|      1|
|          2| 2022-02-15|              1200|2022|      2022-02-15|             2000|      1|
|          3| 2022-03-20|               800|2022|      2022-03-20|             1000|      1|
|          4| 2022-04-10|              3000|2022|      2022-04-10|             2500|      2|
|          5| 2022-05-05|              6000|2022|      2022-05-05|             1800|      2|
+-----------+-----------+------------------+----+----------------+-----------------+-------+
only showing top 5 rows


In [23]:
print("Registros df1:", df1.count())
print("Registros df2:", df2.count())
print("Registros joined_df:", joined_df.count())

Registros df1: 100
Registros df2: 100
Registros joined_df: 100


## 8. Filtrado de transacciones

Se filtran los registros integrados para conservar únicamente aquellas transacciones cuyo `transaction_amount` sea mayor a 1000.

Este paso permite enfocar el análisis posterior en operaciones de mayor valor.

In [24]:
filtered_df = joined_df.filter(
    F.col("transaction_amount") > 1000
)

In [25]:
filtered_df.show(5)

+-----------+-----------+------------------+----+----------------+-----------------+-------+
|customer_id|date_column|transaction_amount|year|transaction_date|transaction_value|quarter|
+-----------+-----------+------------------+----+----------------+-----------------+-------+
|          1| 2022-01-01|              5000|2022|      2022-01-01|             1500|      1|
|          2| 2022-02-15|              1200|2022|      2022-02-15|             2000|      1|
|          4| 2022-04-10|              3000|2022|      2022-04-10|             2500|      2|
|          5| 2022-05-05|              6000|2022|      2022-05-05|             1800|      2|
|          6| 2022-06-10|              4500|2022|      2022-06-10|             1200|      2|
+-----------+-----------+------------------+----+----------------+-----------------+-------+
only showing top 5 rows


In [26]:
print("Registros antes del filtro:", joined_df.count())
print("Registros después del filtro:", filtered_df.count())

Registros antes del filtro: 100
Registros después del filtro: 67


## 9. Agregación del monto total por cliente

A partir de los registros filtrados, se calcula el monto total de transacciones por `customer_id`.

Esta agregación permite resumir el valor acumulado de las operaciones realizadas por cada cliente.

In [27]:
total_amount_per_customer = (
    filtered_df
    .groupBy("customer_id")
    .agg(
        F.sum("transaction_amount").alias("total_transaction_amount")
    )
)

In [28]:
total_amount_per_customer.orderBy(
    F.col("total_transaction_amount").desc()
).show()

+-----------+------------------------+
|customer_id|total_transaction_amount|
+-----------+------------------------+
|          5|                    6000|
|         69|                    5500|
|         93|                    5500|
|         57|                    5500|
|         81|                    5500|
|         45|                    5500|
|         21|                    5500|
|         33|                    5500|
|          1|                    5000|
|         13|                    4800|
|          6|                    4500|
|         27|                    4200|
|         15|                    4200|
|         39|                    4200|
|         87|                    4200|
|         51|                    4200|
|         63|                    4200|
|         75|                    4200|
|         99|                    4200|
|          8|                    3500|
+-----------+------------------------+
only showing top 20 rows


## 10. Persistencia del total por cliente en Hive

El resultado agregado por cliente se almacena en una tabla Hive denominada `customer_totals`.

La persistencia permite conservar el resultado procesado y consultarlo posteriormente mediante Spark SQL sin necesidad de repetir las transformaciones anteriores.

In [30]:
total_amount_per_customer.write \
    .mode("overwrite") \
    .saveAsTable("customer_totals")

26/10/27 03:13:01 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/10/27 03:13:02 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


In [31]:
spark.sql("SHOW TABLES").show()

+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|  default|customer_totals|      false|
+---------+---------------+-----------+



In [32]:
spark.sql("""
    SELECT *
    FROM customer_totals
""").show()

+-----------+------------------------+
|customer_id|total_transaction_amount|
+-----------+------------------------+
|         31|                    3200|
|         85|                    1800|
|         78|                    1500|
|         34|                    1200|
|         81|                    5500|
|         28|                    2600|
|         76|                    2600|
|         27|                    4200|
|         91|                    3200|
|         22|                    1200|
|         93|                    5500|
|          1|                    5000|
|         52|                    2600|
|         13|                    4800|
|          6|                    4500|
|         16|                    2600|
|         40|                    2600|
|         94|                    1200|
|         57|                    5500|
|         54|                    1500|
+-----------+------------------------+
only showing top 20 rows


## 11. Exportación de los datos filtrados en formato Parquet

Los registros que cumplen la condición de `transaction_amount > 1000` se almacenan en HDFS utilizando el formato Parquet.

Parquet es un formato columnar ampliamente utilizado en entornos de procesamiento distribuido debido a su eficiencia de almacenamiento y lectura.

In [33]:
filtered_df.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:8020/data/filtered_data")

In [34]:
spark.read.parquet(
    "hdfs://namenode:8020/data/filtered_data"
).show(5)

+-----------+-----------+------------------+----+----------------+-----------------+-------+
|customer_id|date_column|transaction_amount|year|transaction_date|transaction_value|quarter|
+-----------+-----------+------------------+----+----------------+-----------------+-------+
|          1| 2022-01-01|              5000|2022|      2022-01-01|             1500|      1|
|          2| 2022-02-15|              1200|2022|      2022-02-15|             2000|      1|
|          4| 2022-04-10|              3000|2022|      2022-04-10|             2500|      2|
|          5| 2022-05-05|              6000|2022|      2022-05-05|             1800|      2|
|          6| 2022-06-10|              4500|2022|      2022-06-10|             1200|      2|
+-----------+-----------+------------------+----+----------------+-----------------+-------+
only showing top 5 rows


## 12. Clasificación de transacciones de alto valor

Se crea una nueva variable denominada `high_value` para clasificar las transacciones de acuerdo con su monto.

Las transacciones superiores a 5000 se identifican como `Yes`, mientras que las demás reciben el valor `No`.

In [35]:
df1 = df1.withColumn(
    "high_value",
    F.when(
        F.col("transaction_amount") > 5000,
        F.lit("Yes")
    ).otherwise(
        F.lit("No")
    )
)

In [36]:
df1.select(
    "customer_id",
    "transaction_amount",
    "high_value"
).show(10)

+-----------+------------------+----------+
|customer_id|transaction_amount|high_value|
+-----------+------------------+----------+
|          1|              5000|        No|
|          2|              1200|        No|
|          3|               800|        No|
|          4|              3000|        No|
|          5|              6000|       Yes|
|          6|              4500|        No|
|          7|               200|        No|
|          8|              3500|        No|
|          9|               700|        No|
|         10|              1800|        No|
+-----------+------------------+----------+
only showing top 10 rows


## 13. Promedio del valor de transacción por trimestre

Se calcula el valor promedio de las transacciones para cada trimestre utilizando la variable `quarter` generada anteriormente.

El resultado permite resumir el comportamiento promedio de las transacciones a lo largo de los distintos periodos del año.

In [37]:
average_value_per_quarter = (
    df2
    .groupBy("quarter")
    .agg(
        F.avg("transaction_value").alias("avg_trans_val")
    )
    .orderBy("quarter")
)

In [38]:
average_value_per_quarter.show()

+-------+------------------+
|quarter|     avg_trans_val|
+-------+------------------+
|      1| 1111.111111111111|
|      2|            1072.0|
|      3|1958.3333333333333|
|      4| 816.6666666666666|
+-------+------------------+



## 14. Persistencia de los promedios trimestrales en Hive

El promedio de las transacciones por trimestre se almacena en una segunda tabla Hive denominada `quarterly_averages`.

Esto permite consultar posteriormente los resultados agregados mediante Spark SQL.

In [39]:
average_value_per_quarter.write \
    .mode("overwrite") \
    .saveAsTable("quarterly_averages")

In [40]:
spark.sql("SHOW TABLES").show()

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|  default|   customer_totals|      false|
|  default|quarterly_averages|      false|
+---------+------------------+-----------+



In [41]:
spark.sql("""
    SELECT *
    FROM quarterly_averages
    ORDER BY quarter
""").show()

+-------+------------------+
|quarter|     avg_trans_val|
+-------+------------------+
|      1| 1111.111111111111|
|      2|            1072.0|
|      3|1958.3333333333333|
|      4| 816.6666666666666|
+-------+------------------+



## 15. Total de transacciones por año

Se calcula el monto total de las transacciones agrupado por año.

Esta agregación permite observar el volumen económico acumulado para cada periodo anual disponible en los datos.

In [42]:
total_value_per_year = (
    df1
    .groupBy("year")
    .agg(
        F.sum("transaction_amount")
        .alias("total_transaction_val")
    )
    .orderBy("year")
)

In [43]:
total_value_per_year.show()

+----+---------------------+
|year|total_transaction_val|
+----+---------------------+
|2022|                29800|
|2023|                28100|
|2024|                25700|
|2025|                25700|
|2026|                25700|
|2027|                25700|
|2028|                25700|
|2029|                25700|
|2030|                 9500|
+----+---------------------+



## 16. Exportación del total anual en formato CSV

Finalmente, el monto total de transacciones por año se almacena en HDFS utilizando formato CSV.

Con esta operación se completa el flujo de procesamiento, desde la ingesta de los archivos originales hasta la generación y persistencia de resultados agregados.

In [44]:
total_value_per_year.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("hdfs://namenode:8020/data/total_value_per_year")

In [45]:
spark.read \
    .option("header", True) \
    .csv("hdfs://namenode:8020/data/total_value_per_year") \
    .show()

+----+---------------------+
|year|total_transaction_val|
+----+---------------------+
|2022|                29800|
|2023|                28100|
|2024|                25700|
|2025|                25700|
|2026|                25700|
|2027|                25700|
|2028|                25700|
|2029|                25700|
|2030|                 9500|
+----+---------------------+



## 17. Validación de resultados

Como verificación final, se revisan las tablas almacenadas en Hive y los principales directorios creados en HDFS.

In [46]:
spark.sql("SHOW TABLES").show()

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|  default|   customer_totals|      false|
|  default|quarterly_averages|      false|
+---------+------------------+-----------+



In [47]:
spark.sql("SELECT * FROM customer_totals").show()

spark.sql("""
    SELECT *
    FROM quarterly_averages
    ORDER BY quarter
""").show()

+-----------+------------------------+
|customer_id|total_transaction_amount|
+-----------+------------------------+
|         31|                    3200|
|         85|                    1800|
|         78|                    1500|
|         34|                    1200|
|         81|                    5500|
|         28|                    2600|
|         76|                    2600|
|         27|                    4200|
|         91|                    3200|
|         22|                    1200|
|         93|                    5500|
|          1|                    5000|
|         52|                    2600|
|         13|                    4800|
|          6|                    4500|
|         16|                    2600|
|         40|                    2600|
|         94|                    1200|
|         57|                    5500|
|         54|                    1500|
+-----------+------------------------+
only showing top 20 rows
+-------+------------------+
|quarter| 